# Production API and Backend Patterns — Hands-On

**Software Engineering · Week 04+**

Offline notebook: FastAPI TestClient, Pydantic v2, PyJWT, in-memory SQLite/SQLAlchemy, and OpenTelemetry spans. No network calls.

## 0. Imports and constants

In [ ]:
import base64, hashlib, json, random, time
from collections import defaultdict
from typing import Optional

import jwt
from fastapi import Depends, FastAPI, Header, HTTPException, Query, Response
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, Field
from sqlalchemy import Column, ForeignKey, Integer, String, create_engine, select, update
from sqlalchemy.orm import Session, declarative_base, relationship
from sqlalchemy.pool import StaticPool

SECRET = 'dev-secret'
ISSUER = 'https://issuer.example'
AUDIENCE = 'llm-jobs-api'

## 1. Pydantic v2 contracts and JWT verification

In [ ]:
class JobCreate(BaseModel):
    model_config = ConfigDict(extra='forbid')
    prompt: str = Field(min_length=1, max_length=500)
    tenant_id: str = Field(pattern='^[a-z0-9-]+$')
    client_ref: Optional[str] = Field(default=None, max_length=40)

class JobOut(BaseModel):
    job_id: str
    status: str
    tenant_id: str

class Principal(BaseModel):
    sub: str
    tenant: str
    scopes: set[str]

def mint(scopes='jobs:write jobs:read', tenant='acme'):
    return jwt.encode({'iss': ISSUER, 'sub': 'svc-1', 'tenant': tenant, 'aud': AUDIENCE, 'scope': scopes, 'exp': int(time.time()) + 300}, SECRET, algorithm='HS256')

def verify_token(token: str) -> Principal:
    claims = jwt.decode(token, SECRET, algorithms=['HS256'], audience=AUDIENCE, issuer=ISSUER)
    return Principal(sub=claims['sub'], tenant=claims['tenant'], scopes=set(claims.get('scope', '').split()))

p = verify_token(mint())
print(p.model_dump())

## 2. FastAPI app: auth, rate limit, idempotency, and TestClient

In [ ]:
app = FastAPI(title='Production API Demo')
bearer = HTTPBearer()
rate_counts = defaultdict(int)
idempotency = {}
jobs = {}

async def principal(creds: HTTPAuthorizationCredentials = Depends(bearer)):
    try:
        return verify_token(creds.credentials)
    except jwt.PyJWTError as exc:
        raise HTTPException(401, 'invalid token') from exc

async def require_write(p: Principal = Depends(principal)):
    if 'jobs:write' not in p.scopes:
        raise HTTPException(403, 'missing jobs:write')
    return p

async def rate_limit(response: Response, p: Principal = Depends(require_write)):
    limit = 5
    rate_counts[p.tenant] += 1
    remaining = max(0, limit - rate_counts[p.tenant])
    response.headers['X-RateLimit-Limit'] = str(limit)
    response.headers['X-RateLimit-Remaining'] = str(remaining)
    response.headers['X-RateLimit-Reset'] = '60'
    if rate_counts[p.tenant] > limit:
        raise HTTPException(429, 'rate limited')
    return p

@app.post('/jobs', response_model=JobOut, status_code=202)
def create_job(req: JobCreate, response: Response, idempotency_key: str = Header(..., alias='Idempotency-Key'), p: Principal = Depends(rate_limit)):
    if req.tenant_id != p.tenant:
        raise HTTPException(403, 'wrong tenant')
    body_hash = hashlib.sha256(req.model_dump_json().encode()).hexdigest()
    cache_key = (p.tenant, 'POST', '/jobs', idempotency_key)
    if cache_key in idempotency:
        old_hash, old_response = idempotency[cache_key]
        if old_hash != body_hash:
            raise HTTPException(409, 'idempotency key reused with different body')
        response.headers['Idempotency-Replayed'] = 'true'
        return old_response
    out = JobOut(job_id=f'job-{len(jobs)+1}', status='queued', tenant_id=p.tenant)
    jobs[out.job_id] = out
    idempotency[cache_key] = (body_hash, out)
    return out

client = TestClient(app)
headers = {'Authorization': f'Bearer {mint()}', 'Idempotency-Key': 'idem-1'}
r1 = client.post('/jobs', json={'prompt':'embed docs','tenant_id':'acme'}, headers=headers)
r2 = client.post('/jobs', json={'prompt':'embed docs','tenant_id':'acme'}, headers=headers)
print(r1.status_code, r1.json(), 'replayed?', r2.headers.get('Idempotency-Replayed'))

## 3. Cursor pagination with opaque cursors

In [ ]:
items = [{'id': i, 'created_at': 1000 + i, 'text': f'doc-{i}'} for i in range(1, 8)]

def encode_cursor(created_at, item_id):
    raw = json.dumps({'created_at': created_at, 'id': item_id}).encode()
    return base64.urlsafe_b64encode(raw).decode()

def decode_cursor(cursor):
    if not cursor: return None
    return json.loads(base64.urlsafe_b64decode(cursor.encode()))

def list_items(limit=3, cursor=None):
    after = decode_cursor(cursor)
    rows = items
    if after:
        rows = [r for r in rows if (r['created_at'], r['id']) > (after['created_at'], after['id'])]
    page = rows[:limit]
    next_cursor = encode_cursor(page[-1]['created_at'], page[-1]['id']) if len(page) == limit else None
    return page, next_cursor

page1, cur = list_items()
page2, _ = list_items(cursor=cur)
print([x['id'] for x in page1], [x['id'] for x in page2], cur[:12] + '...')

## 4. ETags and optimistic concurrency

In [ ]:
resource = {'id': 'doc-1', 'body': 'hello', 'version': 1}

def etag(row): return '"v%s"' % row['version']

def update_resource(new_body, if_match):
    if if_match != etag(resource):
        return 412, {'error': 'precondition failed', 'current_etag': etag(resource)}
    resource['body'] = new_body
    resource['version'] += 1
    return 200, {'etag': etag(resource), 'body': resource['body']}

print(update_resource('hello world', '"v1"'))
print(update_resource('lost update attempt', '"v1"'))

## 5. RBAC plus ABAC policy evaluation

In [ ]:
def allowed(principal, action, resource):
    rbac = action in {'jobs:read'} if 'viewer' in principal['roles'] else action in {'jobs:read', 'jobs:write'}
    abac = principal['tenant'] == resource['tenant'] and resource['classification'] <= principal['max_classification']
    return rbac and abac

principal_doc = {'sub':'u1', 'roles':['operator'], 'tenant':'acme', 'max_classification':2}
print('same tenant low class', allowed(principal_doc, 'jobs:write', {'tenant':'acme', 'classification':1}))
print('wrong tenant', allowed(principal_doc, 'jobs:write', {'tenant':'globex', 'classification':1}))

## 6. SQLAlchemy: N+1 shape and joined/batched query

In [ ]:
Base = declarative_base()
class User(Base):
    __tablename__ = 'users'
    id = Column(Integer, primary_key=True)
    name = Column(String)
    posts = relationship('Post')
class Post(Base):
    __tablename__ = 'posts'
    id = Column(Integer, primary_key=True)
    user_id = Column(Integer, ForeignKey('users.id'))
    title = Column(String)

engine = create_engine('sqlite://', connect_args={'check_same_thread': False}, poolclass=StaticPool)
Base.metadata.create_all(engine)
with Session(engine) as s:
    s.add_all([User(id=i, name=f'u{i}') for i in range(1, 4)])
    s.add_all([Post(user_id=u, title=f'p{u}-{j}') for u in range(1, 4) for j in range(2)])
    s.commit()
    users = s.scalars(select(User)).all()
    n_plus_one_queries = 1 + len(users)
    rows = s.execute(select(User.name, Post.title).join(Post, User.id == Post.user_id)).all()
print('n+1 query count shape', n_plus_one_queries, 'joined rows', len(rows))

## 7. Lost update and version-column fix

In [ ]:
accounts = {'a': {'balance': 100, 'version': 1}}
# Two readers observe the same version.
read1 = dict(accounts['a']); read2 = dict(accounts['a'])
accounts['a']['balance'] = read1['balance'] + 10
accounts['a']['balance'] = read2['balance'] + 20
print('lost update balance should be 130 but is', accounts['a']['balance'])

accounts = {'a': {'balance': 100, 'version': 1}}
def update_with_version(delta, expected_version):
    row = accounts['a']
    if row['version'] != expected_version:
        return False
    row['balance'] += delta; row['version'] += 1
    return True
print(update_with_version(10, 1), update_with_version(20, 1), accounts['a'])

## 8. OpenTelemetry retry with exponential backoff and full jitter

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer('api-client')

class FakeAPI:
    def __init__(self, statuses): self.statuses = list(statuses)
    def request(self): return self.statuses.pop(0) if self.statuses else 200

def retry(api, base=0.05, cap=1.0, max_attempts=4, rng=random.Random(8)):
    delays = []
    with tracer.start_as_current_span('rag.submit_job') as root:
        root.set_attribute('tenant', 'acme')
        for attempt in range(max_attempts):
            with tracer.start_as_current_span('http.attempt') as span:
                status = api.request()
                span.set_attribute('retry.attempt', attempt)
                span.set_attribute('http.status_code', status)
            if status not in (429, 502, 503, 504):
                root.set_attribute('final.status_code', status)
                return status, delays
            delays.append(rng.uniform(0, min(cap, base * (2 ** attempt))))
    return status, delays

status, delays = retry(FakeAPI([429, 503, 200]))
print('status', status, 'fake delays', [round(d, 3) for d in delays], 'spans', len(exporter.get_finished_spans()))

## Exercises
1. Add issuer key rotation with a `kid` header and key lookup table.
2. Return `409` when the same Idempotency-Key is reused with a different body.
3. Add `If-None-Match` cache revalidation to the resource example.
4. Add a retry budget so clients stop after cumulative delay exceeds a deadline.

## Links
- Literature note: `02 Literature Notes/Software Engineering/APIs, Integration & Backend Engineering`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 04+ FastAPI JWT Rate Limit Idempotency Demo`, `.../SE Week 04+ OpenTelemetry Retry With Jitter Demo`
- MOC: `06 Maps of Content/Software Engineering Concepts`